In [ ]:
%sql
/* ===================================================================================
   DIAGNOSTIC: Union of Master AUs and CC Mapping
   
   PURPOSE: Combines both lists into a single stacked view using UNION ALL. Each row 
   explicitly indicates if it came from the 'Master AU list' or 'CC'. 
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        'Master AU list' AS Source_Table
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM('US or Canada')) = 'CANADA'
),

CC_Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS BusinessID,
        NULL AS AU_Name, 
        'CC' AS Source_Table
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
)

SELECT BusinessID, AU_Name, Source_Table 
FROM Master_AUs

UNION ALL

SELECT BusinessID, AU_Name, Source_Table 
FROM CC_Mapped_AUs

ORDER BY BusinessID, Source_Table;